# 💻 Chapter 18: Probability in APIs & Rate Limiting
*Track 2 — Developers | Tier 2*

> **🎬 Hook:** How does Netflix decide how many servers to spin up? How does Stripe prevent API abuse? The answer involves Poisson processes and queuing theory.

**🎯 Objectives:** Model API traffic with Poisson processes, understand token bucket rate limiting, size server capacity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# ── Poisson Process: API Requests ──
lambda_rate = 100  # requests per second

# Simulate 10 seconds of API traffic
t_sim = 10
n_requests = rng.poisson(lambda_rate * t_sim)
arrival_times = np.sort(rng.uniform(0, t_sim, n_requests))

# Requests per second
bins = np.arange(0, t_sim+1)
requests_per_sec = np.histogram(arrival_times, bins=bins)[0]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0,0].bar(np.arange(t_sim), requests_per_sec, color='#3498db', edgecolor='white')
axes[0,0].axhline(lambda_rate, color='red', lw=2, linestyle='--', label=f'Expected λ={lambda_rate}')
axes[0,0].set_title('API Requests per Second', fontweight='bold')
axes[0,0].set_xlabel('Second'); axes[0,0].set_ylabel('Requests')
axes[0,0].legend()

# Inter-arrival times (should be Exponential(λ))
inter_arrivals = np.diff(arrival_times)
x = np.linspace(0, 0.1, 300)
axes[0,1].hist(inter_arrivals, bins=50, density=True, color='#3498db', alpha=0.7, label='Observed')
axes[0,1].plot(x, stats.expon.pdf(x, scale=1/lambda_rate), 'r-', lw=2.5,
               label=f'Exp(λ={lambda_rate})')
axes[0,1].set_title('Inter-arrival Times: Exponential Distribution', fontweight='bold')
axes[0,1].set_xlabel('Time between requests (seconds)'); axes[0,1].legend()

# Count distribution (should be Poisson)
k = np.arange(60, 140)
observed_counts = [rng.poisson(lambda_rate) for _ in range(1000)]
axes[1,0].hist(observed_counts, bins=30, density=True, color='#27ae60', alpha=0.7, label='Simulated')
axes[1,0].plot(k, stats.poisson.pmf(k, lambda_rate), 'r-', lw=2.5, label='Poisson(100)')
axes[1,0].set_title('Requests/Second Distribution', fontweight='bold')
axes[1,0].set_xlabel('Requests per second'); axes[1,0].legend()

# Capacity planning: how many servers?
# Each server handles max_rps requests/second
max_rps_per_server = 30
p_overload = np.array([1 - stats.poisson.cdf(n*max_rps_per_server - 1, lambda_rate)
                        for n in range(1, 8)])
axes[1,1].bar(range(1,8), p_overload*100, color='#e74c3c', alpha=0.7)
axes[1,1].axhline(1, color='black', lw=2, linestyle='--', label='1% target')
axes[1,1].set_xlabel('Number of Servers')
axes[1,1].set_ylabel('P(Overload) %')
axes[1,1].set_title(f'Capacity Planning
(each server: {max_rps_per_server} rps max)', fontweight='bold')
axes[1,1].legend()

plt.suptitle("Probability in API Systems: Poisson Traffic Model", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ch18_api_traffic.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"With λ=100 rps, need {next(n for n,p in enumerate(p_overload,1) if p<0.01)+1} servers for <1% overload")

In [ ]:
# ── Token Bucket Rate Limiter ──
class TokenBucket:
    def __init__(self, capacity, refill_rate):
        self.capacity = capacity
        self.tokens = capacity
        self.refill_rate = refill_rate  # tokens/second
        self.last_refill = 0

    def allow_request(self, timestamp, tokens_needed=1):
        # Refill tokens based on elapsed time
        elapsed = timestamp - self.last_refill
        self.tokens = min(self.capacity, self.tokens + elapsed * self.refill_rate)
        self.last_refill = timestamp

        if self.tokens >= tokens_needed:
            self.tokens -= tokens_needed
            return True  # Allowed
        return False  # Rate limited

# Simulate API requests
rng = np.random.default_rng(42)
bucket = TokenBucket(capacity=10, refill_rate=5)  # 5 req/sec steady state, burst of 10

n_requests = 200
request_times = np.cumsum(rng.exponential(0.15, n_requests))  # ~6.7 req/sec
allowed = [bucket.allow_request(t) for t in request_times]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Rate over time
window_s = 1.0
axes[0].scatter(request_times[allowed], np.ones(sum(allowed)),
                color='#27ae60', s=15, alpha=0.7, label='Allowed')
axes[0].scatter(request_times[~np.array(allowed)], np.zeros(sum(~np.array(allowed))),
                color='#e74c3c', s=15, alpha=0.7, label='Rate limited')
axes[0].set_xlabel('Time (seconds)'); axes[0].set_yticks([0,1])
axes[0].set_yticklabels(['Rejected','Allowed'])
axes[0].set_title(f'Token Bucket (capacity=10, rate=5/s)
{sum(allowed)} allowed, {sum(~np.array(allowed))} rejected', fontweight='bold')
axes[0].legend()

# Request rate over time
bin_edges = np.arange(0, request_times[-1]+1, 1)
counts_allowed  = np.histogram(request_times[allowed], bins=bin_edges)[0]
counts_incoming = np.histogram(request_times, bins=bin_edges)[0]
axes[1].bar(bin_edges[:-1], counts_incoming, alpha=0.5, color='gray', label='Incoming')
axes[1].bar(bin_edges[:-1], counts_allowed, alpha=0.8, color='#27ae60', label='Allowed (≤5/s)')
axes[1].axhline(5, color='red', lw=2, linestyle='--', label='Rate limit (5/s)')
axes[1].set_xlabel('Second'); axes[1].set_ylabel('Requests')
axes[1].set_title('Effective Rate Limiting', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('ch18_rate_limiting.png', dpi=150, bbox_inches='tight')
plt.show()

## ✏️ Section 6 — Coding Challenges

**Challenge 1:** Model: API receives Poisson(50 req/sec). Each takes Exponential(1/10 sec) to process.
Using Little's Law: L = λW, compute average queue length and wait time.

**Challenge 2:** Implement a "leaky bucket" rate limiter and compare to token bucket.

**Challenge 3:** Build a simple load balancer that routes to the server with fewest active requests, simulating request arrivals and completions.

<details><summary>Solutions</summary>

**C1:** λ=50, service rate μ=10, utilization ρ=λ/μ=5 (system overloaded at ρ>1!).
Need multiple servers: for M/M/k, ρ<1 requires k>λ/μ=5, so k≥6 servers.

**C2:** Leaky bucket vs token bucket differ in how they handle bursts.
</details>

In [ ]:
# M/M/1 Queue Analysis (basic queuing theory)
def mm1_stats(arrival_rate, service_rate):
    rho = arrival_rate / service_rate  # utilization
    if rho >= 1:
        return None  # unstable
    L  = rho / (1 - rho)       # avg customers in system
    Lq = rho**2 / (1 - rho)    # avg customers in queue
    W  = 1 / (service_rate - arrival_rate)  # avg time in system
    Wq = rho / (service_rate - arrival_rate) # avg wait in queue
    return {'rho': rho, 'L': L, 'Lq': Lq, 'W': W, 'Wq': Wq}

print("M/M/1 Queue: API Server Analysis")
for lam in [5, 8, 9, 9.5]:
    mu = 10
    r = mm1_stats(lam, mu)
    if r:
        print(f"  λ={lam:>4}/s: ρ={r['rho']:.2f}, avg wait={r['Wq']:.3f}s, queue len={r['Lq']:.2f}")
    else:
        print(f"  λ={lam:>4}/s: UNSTABLE (ρ≥1)")
print()
print("💡 As utilization approaches 1 (100%), wait time → ∞!")
print("   This is why engineers keep utilization below 70-80%.")

## 🎯 Recap
1. API traffic ≈ Poisson arrivals; service times ≈ Exponential.
2. Token bucket smooths burst traffic; Little's Law links queue length, rate, and wait time.
3. At high utilization (ρ → 1), queues and latency blow up — plan for headroom.

**Next:** [Chapter 19 — Randomized Algorithms]